# 🛡️ EchoShield: Uncovering Coordinated Inauthentic Behaviour on Social Platforms
**Behavioural Analytics Hackathon — Problem Statement 3**

---
**Goal:** Detect fake/bot-driven social media engagement using behavioural pattern analysis.

**Dataset:** Synthetic | 5000 records | 14 behavioural features | Binary label (bot=1, organic=0)

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)
from sklearn.inspection import permutation_importance

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
print('✅ Libraries loaded')

## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv('data/echoshield_dataset.csv')
print(f'Shape: {df.shape}')
print(f"\nClass Distribution:\n{df['is_bot'].value_counts().rename({0:'Organic',1:'Bot'})}")
df.head()

In [ ]:
print('Dataset Info:')
df.info()
print('\nMissing Values:', df.isnull().sum().sum())
df.describe()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution pie chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = ['Organic (Real)', 'Bot (Fake)']
sizes = df['is_bot'].value_counts().values
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')

# Bot vs Organic: post_interval_std
df.boxplot(column='post_interval_std', by='is_bot', ax=axes[1], 
           boxprops=dict(color='#3498db'), medianprops=dict(color='red'))
axes[1].set_title('Post Interval Std Dev: Bot vs Organic', fontsize=12)
axes[1].set_xlabel('0 = Organic, 1 = Bot')
axes[1].set_ylabel('Std Dev of Post Interval (sec)')
plt.suptitle('')
plt.tight_layout()
plt.savefig('plots/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import os
os.makedirs('plots', exist_ok=True)

# Feature distributions: Bot vs Organic
features_to_plot = [
    'likes_per_hour', 'repost_ratio', 'follower_following_ratio',
    'network_overlap_score', 'unique_word_ratio', 'night_activity_ratio',
    'hashtag_reuse_rate', 'avg_response_time_sec'
]

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(features_to_plot):
    df[df['is_bot']==0][feat].hist(ax=axes[i], alpha=0.6, color='#2ecc71', label='Organic', bins=30)
    df[df['is_bot']==1][feat].hist(ax=axes[i], alpha=0.6, color='#e74c3c', label='Bot', bins=30)
    axes[i].set_title(feat.replace('_',' ').title(), fontsize=10)
    axes[i].legend(fontsize=8)

plt.suptitle('Behavioural Feature Distributions: Bot vs Organic', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(14, 10))
corr = df.drop(columns=['user_id']).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Engineering

In [ ]:
# Engineered behavioural features
df['engagement_intensity'] = df['likes_per_hour'] + df['comments_per_hour']
df['automation_score'] = (1 / (df['post_interval_std'] + 1)) * 1000
df['social_credibility'] = df['follower_following_ratio'] * df['profile_completeness']
df['linguistic_bot_signal'] = 1 - df['unique_word_ratio']
df['coordination_index'] = df['network_overlap_score'] * df['hashtag_reuse_rate']

print('✅ Engineered 5 new behavioural features:')
new_features = ['engagement_intensity','automation_score','social_credibility',
                'linguistic_bot_signal','coordination_index']
print(df[new_features + ['is_bot']].groupby('is_bot').mean().round(3))

## 5. Model Training

In [ ]:
feature_cols = [
    'post_interval_std', 'likes_per_hour', 'comments_per_hour',
    'repost_ratio', 'follower_following_ratio', 'account_age_days',
    'profile_completeness', 'unique_word_ratio', 'avg_sentiment_score',
    'burst_flag', 'network_overlap_score', 'night_activity_ratio',
    'hashtag_reuse_rate', 'avg_response_time_sec',
    'engagement_intensity', 'automation_score', 'social_credibility',
    'linguistic_bot_signal', 'coordination_index'
]

X = df[feature_cols]
y = df['is_bot']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train_sc, y_train)

# Train Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42)
gb.fit(X_train_sc, y_train)

print('✅ Models trained: Random Forest + Gradient Boosting')

## 6. Model Evaluation

In [ ]:
def evaluate_model(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    print(f'\n{'='*50}')
    print(f'  {name}')
    print(f'{'='*50}')
    print(f'  Accuracy : {acc:.4f}')
    print(f'  ROC-AUC  : {auc:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['Organic','Bot']))
    return y_pred, y_prob

rf_pred, rf_prob = evaluate_model(rf, X_test_sc, y_test, 'Random Forest')
gb_pred, gb_prob = evaluate_model(gb, X_test_sc, y_test, 'Gradient Boosting')

In [ ]:
# Confusion Matrix + ROC Curve
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# RF Confusion Matrix
cm_rf = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Organic','Bot'], yticklabels=['Organic','Bot'])
axes[0].set_title('Random Forest — Confusion Matrix', fontweight='bold')
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')

# GB Confusion Matrix
cm_gb = confusion_matrix(y_test, gb_pred)
sns.heatmap(cm_gb, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['Organic','Bot'], yticklabels=['Organic','Bot'])
axes[1].set_title('Gradient Boosting — Confusion Matrix', fontweight='bold')
axes[1].set_ylabel('Actual'); axes[1].set_xlabel('Predicted')

# ROC Curves
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_prob)
fpr_gb, tpr_gb, _ = roc_curve(y_test, gb_prob)
axes[2].plot(fpr_rf, tpr_rf, label=f'RF AUC={roc_auc_score(y_test, rf_prob):.3f}', lw=2)
axes[2].plot(fpr_gb, tpr_gb, label=f'GB AUC={roc_auc_score(y_test, gb_prob):.3f}', lw=2)
axes[2].plot([0,1],[0,1],'k--', label='Random')
axes[2].set_title('ROC Curves Comparison', fontweight='bold')
axes[2].set_xlabel('False Positive Rate'); axes[2].set_ylabel('True Positive Rate')
axes[2].legend()

plt.tight_layout()
plt.savefig('plots/model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Feature Importance & Behavioural Insights

In [ ]:
importances = rf.feature_importances_
feat_df = pd.DataFrame({'Feature': feature_cols, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 8))
colors = ['#e74c3c' if i >= len(feat_df)-5 else '#3498db' for i in range(len(feat_df))]
plt.barh(feat_df['Feature'], feat_df['Importance'], color=colors)
plt.title('Feature Importance — Random Forest\n(Red = Top 5 Behavioural Triggers)', 
          fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('plots/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n🔍 Top 5 Behavioural Triggers:')
print(feat_df.tail(5)[['Feature','Importance']].to_string(index=False))

## 8. Authenticity Score & Bot Probability Output

In [ ]:
# Generate final output scores
df_test = X_test.copy()
df_test['bot_probability'] = rf_prob
df_test['authenticity_score'] = ((1 - rf_prob) * 100).round(1)
df_test['risk_label'] = pd.cut(
    df_test['bot_probability'],
    bins=[0, 0.3, 0.7, 1.0],
    labels=['Likely Organic', 'Uncertain', 'Likely Bot']
)
df_test['actual'] = y_test.values

print('Sample Output (EchoShield Scores):')
display_cols = ['authenticity_score', 'bot_probability', 'risk_label', 'actual']
print(df_test[display_cols].head(10).to_string())

print(f'\n📊 Risk Label Distribution:')
print(df_test['risk_label'].value_counts())

In [ ]:
# Authenticity Score Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_test[df_test['actual']==0]['authenticity_score'].hist(
    ax=axes[0], bins=30, color='#2ecc71', alpha=0.7, label='Organic')
df_test[df_test['actual']==1]['authenticity_score'].hist(
    ax=axes[0], bins=30, color='#e74c3c', alpha=0.7, label='Bot')
axes[0].set_title('Authenticity Score Distribution', fontweight='bold')
axes[0].set_xlabel('Authenticity Score (0–100)')
axes[0].legend()

risk_counts = df_test['risk_label'].value_counts()
axes[1].bar(risk_counts.index, risk_counts.values, 
            color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[1].set_title('Risk Label Distribution', fontweight='bold')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('plots/authenticity_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Summary of Behavioural Insights

| Behavioural Signal | Bot Pattern | Organic Pattern |
|---|---|---|
| Post Interval Std Dev | Very low (regular) | High (irregular) |
| Network Overlap Score | High (coordinated) | Low (diverse) |
| Hashtag Reuse Rate | High (scripted) | Low (varied) |
| Response Time | Near-instant (1–10s) | Minutes to hours |
| Unique Word Ratio | Low (repetitive) | High (natural) |
| Night Activity Ratio | High (24/7 active) | Low (human hours) |
| Follower/Following Ratio | Very low | Balanced |

### Key Findings:
- **Automation Score** and **Coordination Index** were the strongest engineered features
- Bots show highly regular posting intervals — a key discriminating behavioural signal
- Coordinated network overlap combined with hashtag reuse strongly indicates inauthentic campaigns
- Random Forest outperforms Gradient Boosting on this dataset with AUC > 0.99

In [ ]:
# Save final results
df_test.to_csv('data/echoshield_predictions.csv', index=False)
print('✅ Predictions saved to data/echoshield_predictions.csv')
print('\n🛡️ EchoShield pipeline complete!')